In [1]:
from tedge import tGraph, tGraphNE, METHOD_MAP, linear_rank_mapping, weight_choice, random_seed, load_labels
import networkx as nx
import scipy.sparse as sp
import numpy as np
import pprint
from gensim.models import Word2Vec
import pandas as pd
import random
from numba import jit, njit
from copy import deepcopy as dc

In [2]:
def load_json(filename):
    if 'npy' not in filename:
        filename += '.npy'
    return np.load(filename, allow_pickle=True).item()
js = load_json('result/test_edges/bc1_2022_01_20_23_57_24_3.npy')

In [ ]:
[[8738, 4750, 7274, 8971, 5097, 9629, 997, 8517, 6779, 5150, 200, 7090, 5073, 7493, 9035, 8540]]
acc:0.6470588235294118
[Parameter containing:
tensor([[ 0.2651, -0.3050, -0.1155, -0.0981, -0.3454, -0.2875],
        [-0.3323,  0.4742, -0.2443, -0.0763,  0.3128, -0.1375],
        [ 0.1400, -0.4605,  0.2474, -0.4996,  0.0277, -0.1367],
        [ 0.2916,  0.1897, -0.0253,  0.0111, -0.0854,  0.0528],
        [-0.1624,  0.1593,  0.2066,  0.1197, -0.5107,  0.3477],
        [ 0.3799,  0.2503,  0.1461,  0.3119, -0.0474,  0.2648],
        [ 0.1584, -0.2181,  0.1962,  0.3420, -0.3107, -0.0223],
        [-0.1787, -0.2690,  0.1568, -0.3608,  0.0378,  0.0341],
        [ 0.0757,  0.0364, -0.1046, -0.2299, -0.1338,  0.4247],
        [-0.2072, -0.1739,  0.1994, -0.0515, -0.2382, -0.1292],
        [ 0.3430,  0.5738, -0.1850,  0.1922, -0.0724,  0.5680],
        [-0.2735,  0.1818, -0.3589, -0.3891,  0.2373,  0.1084],
        [ 0.2229, -0.0362, -0.2083,  0.0402, -0.2414,  0.0879],
        [-0.0244,  0.1813, -0.0090, -0.0298, -0.0164, -0.0028],
        [-0.0067, -0.0535,  0.2629, -0.0528, -0.0341, -0.1604],
        [-0.1849, -0.1975,  0.0470, -0.0890,  0.0732, -0.2448]],
       requires_grad=True), Parameter containing:
tensor([-0.1293, -0.3675,  0.0650,  0.0032,  0.2927, -0.2878,  0.0343,  0.2356,
         0.1336, -0.3328, -0.1871, -0.0819, -0.3583, -0.0490, -0.3370,  0.2108],
       requires_grad=True), Parameter containing:
tensor([[ 0.1145, -0.0952,  0.1335, -0.0377, -0.1378,  0.0376,  0.1154,  0.3958,
          0.2233,  0.1759, -0.1772, -0.2619, -0.0270,  0.0057,  0.0210, -0.0849],
        [-0.0731,  0.1008, -0.3653, -0.1133, -0.0244,  0.0048,  0.2188, -0.3416,
          0.1268,  0.0876, -0.1173,  0.1857,  0.0928, -0.0191,  0.1675,  0.0540]],
       requires_grad=True), Parameter containing:
tensor([ 0.3173, -0.2654], requires_grad=True)]

In [3]:
js.keys()

dict_keys(['seed', 'sga', 'MLP', 'SGC2', 'GCN2', 'FastGCN'])

In [12]:
js['sga'].keys()

dict_keys([])

In [21]:
del js['seed']

In [23]:
js.keys()

dict_keys(['sga_TEDGE', 'sga_TBS', 'sga_WBS', 'sga_TBS+WBS', 'MLP_TEDGE', 'MLP_TBS', 'MLP_WBS', 'MLP_TBS+WBS', 'SGC2_TEDGE', 'SGC2_TBS', 'SGC2_WBS', 'SGC2_TBS+WBS', 'GCN2_TEDGE', 'GCN2_TBS', 'GCN2_WBS', 'GCN2_TBS+WBS', 'FastGCN_TEDGE', 'FastGCN_TBS', 'FastGCN_WBS', 'FastGCN_TBS+WBS'])

In [17]:
list(js.keys() - set([ 'sga_TBS', 'sga_WBS']))

['GCN2_TEDGE',
 'FastGCN_WBS',
 'GCN2_TBS',
 'FastGCN_TBS',
 'MLP_TBS',
 'FastGCN_TEDGE',
 'GCN2_WBS',
 'seed',
 'SGC2_TEDGE',
 'FastGCN_TBS+WBS',
 'SGC2_TBS+WBS',
 'GCN2_TBS+WBS',
 'sga_TEDGE',
 'MLP_WBS',
 'SGC2_TBS',
 'SGC2_WBS',
 'MLP_TEDGE',
 'sga_TBS+WBS',
 'MLP_TBS+WBS']

In [15]:
set([ 'sga_TBS', 'sga_WBS'])

{'sga_TBS', 'sga_WBS'}

In [4]:
np.diff([1]).mean()

D:\anaconda\envs\tt\lib\site-packages\ipykernel_launcher.py:1: RuntimeWarning: Mean of empty slice.
  """Entry point for launching an IPython kernel.
D:\anaconda\envs\tt\lib\site-packages\numpy\core\_methods.py:170: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


nan

In [12]:
import os

In [8]:
list(pd.read_csv('result/test_edges/tedge_2022_01_05_18_09_43_0_0.csv', index_col=0).index)

['sga_SVM_TEDGE', 'sga_SVM_TBS']

In [178]:
class ARGS():
    def __init__(self):
        self.tedge_type = "TBS"
        self.dimensions = 12
        self.num_walks = 1
        self.walk_length = 5
        self.window_size = 4
        self.workers = 8
        self.train_size = 0.5
        self.verbose = 0
        self.is_dan = True
args = ARGS()
args.time_biased_type, args.first_biased_type, args.amount_biased, args.alpha = METHOD_MAP[args.tedge_type]

In [246]:
tG = tGraph('dataset/phishing/TransEdgelist.txt', verbose=1)

Loading file dataset/phishing/TransEdgelist.txt ...
Summary of graph:
Number of nodes:  86622
Number of edges:  106083
Number of edge_key:  401176
Min time:  1438923532


In [4]:
tGNE = tGraphNE(tG, args.time_biased_type, args.first_biased_type, args.amount_biased, args.alpha,
                    dimensions=args.dimensions, num_walks=args.num_walks,
                    walk_length=args.walk_length, window_size=args.window_size,
                    workers=args.workers, output='test')

In [7]:
embedding = tGNE.word2vec_model.wv.vectors[np.fromiter(
                map(int, tGNE.word2vec_model.wv.index_to_key), np.int32).argsort()]

In [8]:
embedding

array([[ 0.03153495, -0.06492371, -0.163023  , ...,  0.04727796,
        -0.20014933, -0.06593096],
       [ 0.40657353,  0.1752386 ,  0.26357788, ..., -0.07565748,
        -0.00908539, -0.05857573],
       [-0.02399746, -0.16234699, -0.10513218, ...,  0.17199215,
         0.08704875, -0.15775238],
       ...,
       [-0.07518569,  0.06889989,  0.01763538, ...,  0.08138565,
        -0.07632921, -0.03810624],
       [-0.09802724,  0.06100797,  0.05047184, ..., -0.03527083,
         0.04392264,  0.02984988],
       [ 0.2723489 , -0.2046812 , -0.2126417 , ...,  0.27045596,
        -0.09277388,  0.1232906 ]], dtype=float32)

In [9]:
tGNE.features

array([[ 0.03153495, -0.06492371, -0.163023  , ...,  0.04727796,
        -0.20014933, -0.06593096],
       [ 0.40657353,  0.1752386 ,  0.26357788, ..., -0.07565748,
        -0.00908539, -0.05857573],
       [-0.02399746, -0.16234699, -0.10513218, ...,  0.17199215,
         0.08704875, -0.15775238],
       ...,
       [-0.07518569,  0.06889989,  0.01763538, ...,  0.08138565,
        -0.07632921, -0.03810624],
       [-0.09802724,  0.06100797,  0.05047184, ..., -0.03527083,
         0.04392264,  0.02984988],
       [ 0.2723489 , -0.2046812 , -0.2126417 , ...,  0.27045596,
        -0.09277388,  0.1232906 ]], dtype=float32)

In [247]:
G = tG.G

In [248]:
g = nx.DiGraph(G)

In [272]:
adj = nx.to_scipy_sparse_matrix(g)

In [273]:
adj.data = np.ones(len(adj.data))

In [285]:
indices = adj.indices
indptr = adj.indptr

In [310]:
u = 0
u_out_nbrs = indices[indptr[u]:indptr[u + 1]]
u_in_nbrs = adj[:, u].nonzero()[0]

In [311]:
u_out_nbrs

array([    1,  1083,  1106,  1154,  1156,  1163,  1412,  1926,  1949,
        2292,  2332,  2476,  2860,  2983,  3213,  3409,  3483,  3724,
        4044,  4048,  4276,  6580,  9592,  9840, 10353, 13892, 14424,
       14491, 14686, 14981, 19347, 19355, 19357, 19746, 21702, 22169,
       25195, 25325, 25881, 25933, 25963, 26039, 26276, 26304, 26386,
       26615, 30240, 31025, 32314, 34862, 34910, 35232, 35254, 35820,
       36212, 36463, 36879, 40112, 41145, 41185, 41709, 42016, 42360,
       42482, 42488, 43745, 43995, 44734, 45330, 46222, 46272, 46293,
       46620, 46660, 46697, 46713, 47287, 47588, 48745, 48746, 48770,
       48931, 49031, 49041, 53722, 53832, 53998, 54138, 54372, 54998,
       59179, 59275, 64064, 64184, 64294, 65498, 65511, 66011, 66034,
       66389, 66524, 66813, 67636, 67994, 68370, 68638, 68957, 69037,
       69239, 69516, 69560, 70612, 70692, 71025, 72012, 73203, 73339,
       73520, 73723, 74688, 74699, 75157, 75173, 75196, 77831, 78290,
       78465, 78533,

In [312]:
u_in_nbrs

array([ 2080,  2084,  3133,  3756,  6584, 10355, 18119, 19356, 20127,
       21649, 44245, 47993, 47995, 48372, 48854, 53923, 59066, 59175,
       64067, 68644, 69028, 69034, 75161, 78911, 79246, 79827, 80292,
       80758, 81771, 85413, 85418])

In [318]:
his_out_amount = amount_data[u, u_out_nbrs].toarray()[0].sum()
his_in_amount = amount_data[u_in_nbrs, u].toarray()[1].sum()

In [315]:
his_out_amount

3641.9369849800005

In [319]:
his_in_amount

0.99951826

In [316]:
his_in_amount = np.sum([amount_data[u_in_nbr, u] for u_in_nbr in u_in_nbrs])

In [307]:
print(adj[1:5:,1].nonzero())

(array([1, 2, 3]), array([0, 0, 0]))


In [308]:
adj[0,1:5].toarray()

array([[1., 0., 0., 0.]])

In [274]:
print(adj)

  (0, 1)	1.0
  (0, 1083)	1.0
  (0, 1106)	1.0
  (0, 1154)	1.0
  (0, 1156)	1.0
  (0, 1163)	1.0
  (0, 1412)	1.0
  (0, 1926)	1.0
  (0, 1949)	1.0
  (0, 2292)	1.0
  (0, 2332)	1.0
  (0, 2476)	1.0
  (0, 2860)	1.0
  (0, 2983)	1.0
  (0, 3213)	1.0
  (0, 3409)	1.0
  (0, 3483)	1.0
  (0, 3724)	1.0
  (0, 4044)	1.0
  (0, 4048)	1.0
  (0, 4276)	1.0
  (0, 6580)	1.0
  (0, 9592)	1.0
  (0, 9840)	1.0
  (0, 10353)	1.0
  :	:
  (86454, 86482)	1.0
  (86456, 86511)	1.0
  (86456, 86512)	1.0
  (86456, 86513)	1.0
  (86456, 86514)	1.0
  (86459, 59)	1.0
  (86460, 59)	1.0
  (86461, 59)	1.0
  (86462, 19204)	1.0
  (86462, 86487)	1.0
  (86462, 86488)	1.0
  (86462, 86489)	1.0
  (86463, 59)	1.0
  (86464, 59)	1.0
  (86465, 59)	1.0
  (86466, 59)	1.0
  (86467, 59)	1.0
  (86468, 59)	1.0
  (86469, 55)	1.0
  (86469, 2727)	1.0
  (86469, 26640)	1.0
  (86469, 86608)	1.0
  (86469, 86609)	1.0
  (86469, 86610)	1.0
  (86469, 86611)	1.0


In [113]:
oadj = adj + adj.T


In [112]:
print(oadj)

1


In [117]:
oadj.data = np.ones(oadj.data.shape[0])

In [118]:
oadj.data

array([1., 1., 1., ..., 1., 1., 1.])

In [119]:
oadj

<86622x86622 sparse matrix of type '<class 'numpy.float64'>'
	with 208644 stored elements in Compressed Sparse Row format>

In [102]:
adj.sum(0).A1

array([5.52863161e+03, 1.22973500e+02, 9.50000000e-02, ...,
       4.00000000e-04, 2.00000000e-03, 7.81354000e-03])

In [77]:
adj.sum(axis=1)

matrix([[3641.93698498],
        [  24.54086404],
        [3702.382478  ],
        ...,
        [   0.        ],
        [   0.        ],
        [   0.        ]])

In [253]:
slabels = load_labels('dataset/phishing/label.txt') # 890

In [254]:
N = adj.shape[0]
labels = np.zeros(N).astype('int32')
for k, v in slabels.items():
    if v == 1:
        labels[np.int32(k)] = 1

In [255]:
N = adj.shape[0]
amount_data = sp.lil_matrix((N, N), dtype=np.float64)
timestamp_data = sp.lil_matrix((N, N), dtype=np.int64)
amount_timestamp_data = sp.lil_matrix((N, N), dtype=np.float64)
alpha = 0.5
for i in range(N):
    cur = str(i)
    nbrs = list(G.neighbors(cur))
    for nbr in nbrs:
        latest_timestamp = list(G.get_edge_data(cur, nbr))
        amount = G[cur][nbr][latest_timestamp[-1]]['weight']
        amount_data[i, int(nbr)] = amount
        timestamp_data[i, int(nbr)] = len(latest_timestamp)
        amount_timestamp_data[i, int(nbr)] = (amount**alpha) * (len(latest_timestamp)**(1-alpha))
amount_data = amount_data.tocsr()
timestamp_data = timestamp_data.tocsr()
amount_timestamp_data = amount_timestamp_data.tocsr()

In [284]:
amount_data[0,1]

16.55

In [277]:
adj.copy()

<86622x86622 sparse matrix of type '<class 'numpy.float64'>'
	with 106083 stored elements in Compressed Sparse Row format>

In [75]:
amount_timestamp_data.tolil().nonzero()

(array([    0,     0,     0, ..., 86469, 86469, 86469], dtype=int32),
 array([    1,  1083,  1106, ..., 86609, 86610, 86611], dtype=int32))

In [276]:
np.savez('C://Users/pc/Desktop/trans2vec.npz', adj_matrix=adj, amount_data=amount_data, timestamp_data=timestamp_data, node_label=labels)

In [39]:
data = np.load('C://Users/pc/Desktop/tedge_trans2vec.npz', allow_pickle=True)

In [41]:
data['adj_matrix'].item()

<86622x86622 sparse matrix of type '<class 'numpy.float64'>'
	with 106083 stored elements in Compressed Sparse Row format>

In [105]:
pd.DataFrame(labels).to_csv('result/test_tedge/label.csv', index=None)

In [21]:
# pd.read_csv('dataset/phishing/test_tedge/label.csv').values.ravel()

In [42]:
def load_embeddings(filename):
    fin = open(filename, 'r')
    node_num, size = [int(x) for x in fin.readline().strip().split()]
    vectors = {}
    while 1:
        l = fin.readline()
        if l == '':
            break
        vec = l.strip().split(' ')
        assert len(vec) == size + 1
        vectors[vec[0]] = [float(x) for x in vec[1:]]
    fin.close()
    assert len(vectors) == node_num, "load_embeddings error"
    return vectors

In [44]:
embeddings = load_embeddings('result/test_tedge/TBS_2022_01_04_14_34_00_0')

In [41]:
np.savez('C://Users/pc/Desktop/tedge.npz', adj_matrix=adj, node_attr=np.zeros((N,1)), node_label=labels)

In [43]:
tGNE = tGraphNE(tG, args.time_biased_type, args.first_biased_type, args.amount_biased, args.alpha,
                    dimensions=args.dimensions, num_walks=args.num_walks,
                    walk_length=args.walk_length, window_size=args.window_size,
                    workers=args.workers, output='test')

In [44]:
w2v = tGNE.word2vec_model

AttributeError: 'tGraphNE' object has no attribute 'word2vec_model'

In [13]:
pprint.pprint (vars(w2v.wv).keys())

dict_keys(['vector_size', 'index_to_key', 'next_index', 'key_to_index', 'vectors', 'norms', 'expandos', 'mapfile_path', 'vectors_lockf'])


In [26]:
index_to_key = np.array(w2v.wv.index_to_key).astype(int)

In [68]:
tup = sorted(zip(vectors, index_to_key), key=lambda x: x[1], reverse=False)

In [69]:
features = np.array([t[0] for t in tup])

In [84]:
import pandas as pd
pd.DataFrame(features).to_csv('result/test_tedge/test.csv', index=None)

In [119]:
pd.read_csv('result/test_tedge/TBS_2022_01_04_17_14_52_test.csv')

,Unnamed: 0,0,1,2,3,4,5,6,7,8,9,10,11
0,0,0.250770,-0.052570,-0.204898,-0.331907,-0.049654,0.548962,0.032967,0.100340,-0.165145,-0.077892,0.348065,-0.313526
1,1,0.211098,-0.424510,0.134299,-0.136596,-0.324925,-0.452726,0.266274,-0.280284,-0.155689,-0.201081,0.135063,-0.050097
2,2,0.155993,0.118773,-0.114414,0.071476,0.104470,0.150146,0.235364,-0.227806,-0.017682,0.213925,-0.056307,0.112784
3,3,-0.094241,-0.077669,-0.104865,-0.075431,-0.063374,0.140069,0.055479,-0.053595,0.009217,-0.001424,-0.112932,0.077431
4,4,-0.127655,-0.271294,0.003652,0.047367,0.106174,0.167985,0.038318,-0.143544,0.031077,-0.043049,-0.259341,-0.163098
...,...,...,...,...,...,...,...,...,...,...,...,...,...
86617,86617,-0.043935,-0.026977,0.049618,-0.033054,0.012348,-0.001135,-0.059755,0.045042,0.034653,-0.032077,0.003371,0.053196
86618,86618,0.003024,-0.017248,-0.040953,0.016994,0.025792,0.067622,0.017139,0.028307,0.056494,-0.047176,-0.036733,-0.037177
86619,86619,0.233256,-0.060064,-0.137907,-0.156309,0.164418,-0.077807,0.068971,-0.011457,-0.208537,0.089438,0.090449,0.014230
86620,86620,0.030206,0.008975,0.061862,0.043683,0.000417,0.048638,0.065510,0.019710,0.001632,-0.070690,0.055847,0.008467


In [117]:
features.shape

(86622, 12)

In [110]:
sample_labels = load_labels('dataset/phishing/label.txt') # 890

In [144]:
nodes = list([int(node) for node in sample_labels.keys()])
nodes_labels = list(sample_labels.values())
nodes_embeddings = features[nodes]
# print(nodes_embeddings)
# print(nodes_labels)
nodes_embeddings = pd.DataFrame(nodes_embeddings, index=nodes)

In [146]:
X_train, X_test, y_train, y_test = train_test_split(nodes_embeddings, nodes_labels, train_size=args.train_size, random_state=2022)
model = SVC(kernel='linear', C=0.4, random_state=2022)
model.fit(X_train, y_train)

SVC(C=0.4, kernel='linear', random_state=2022)

In [151]:
model.predict(features)

array([1, 0, 0, ..., 0, 0, 0])

In [130]:
y_pred = model.predict(features)
roc = roc_auc_score(y_test, y_pred)
apc = average_precision_score(y_test, y_pred)
cr = classification_report(y_test, y_pred)
print('roc_auc_score:{}'.format(roc))
print('average_precision_score:{}'.format(apc))
print('classification_report:\n{}'.format(cr))

roc_auc_score:0.7472808507197153
average_precision_score:0.7144229026637923
classification_report:
              precision    recall  f1-score   support

           0       0.69      0.87      0.77       216
           1       0.83      0.63      0.72       229

    accuracy                           0.74       445
   macro avg       0.76      0.75      0.74       445
weighted avg       0.76      0.74      0.74       445



In [120]:
embeddings = pd.read_csv(output).values # 8w6+
# labels = pd.read_csv('dataset/phishing/label.csv').values.ravel()
sample_labels = load_labels('dataset/phishing/label.txt') # 890
nodes = list([int(node) for node in sample_labels.keys()])
nodes_labels = list(sample_labels.values())
nodes_embeddings = embeddings[nodes]

X_train, X_test, y_train, y_test = train_test_split(nodes_embeddings, nodes_labels, train_size=args.train_size, random_state=args.seed)
model = SVC(kernel='linear', C=0.4, random_state=args.seed)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
roc = roc_auc_score(y_test, y_pred)
apc = average_precision_score(y_test, y_pred)
cr = classification_report(y_test, y_pred)
print('roc_auc_score:{}'.format(roc))
print('average_precision_score:{}'.format(apc))
print('classification_report:\n{}'.format(cr))

NameError: name 'output' is not defined

In [ ]:
丹总的文章数据集用的是445个钓鱼节点以及445个随机选择的钓鱼节点，但网络嵌入时用的是者890个节点采样出来的图，有8.6w+个点；
文章做分类任务时，是用学习到的特征输入到svm中做分类；
1. 替代模型用SGC，数据集用8.6w+个点，特征为单位矩阵
2. 替代模型用SGC，数据集用890个点，特征为丹总的嵌入方法得到的矩阵的子集（890个点）
3. 替代模型用SGC，数据集用8.6w+个点，特征为丹总的嵌入方法得到的矩阵

算出扰动边后，插入到数据集中，重新跑一遍嵌入方法得到扰动矩阵；对比前后两个特征矩阵输入到svm的分类准确率

In [2]:
def load_json(filename):
    if 'npy' not in filename:
        filename += '.npy'
    return np.load(filename, allow_pickle=True).item()

In [5]:
edges_flips = load_json('result/test_edges/tedge_2022_01_05_18_09_43_0.npy')

In [17]:
list(set([key.split('_')[-1] for key in edges_flips.keys()]) - {'seed'})

['TBS', 'TBS+WBS', 'TEDGE', 'WBS']

In [112]:
list(set(edges_flips.keys()) - {'seed'})

['FastGCN_TEDGE',
 'FastGCN_TBS+WBS',
 'FastGCN_WBS',
 'sga_TBS',
 'SGC2_TBS+WBS',
 'MLP_TBS',
 'sga_TEDGE',
 'sga_TBS+WBS',
 'SGC2_TBS',
 'MLP_TBS+WBS',
 'GCN2_TBS',
 'GCN2_TEDGE',
 'FastGCN_TBS',
 'SGC2_TEDGE',
 'sga_WBS',
 'SGC2_WBS',
 'MLP_TEDGE',
 'GCN2_WBS',
 'GCN2_TBS+WBS',
 'MLP_WBS']

In [21]:
from sklearn.svm import SVC

In [23]:
A = SVC()

In [25]:
A.name = "SVC"

In [34]:
def get_adj_flips(edge_flips):
    flips = edge_flips
    if flips is None or len(flips) == 0:
        return None

    if isinstance(flips, dict):
        flips = list(flips.keys())

    return np.asarray(flips, dtype="int64")

In [114]:
for eee in list(set(edges_flips.keys()) - {'seed'}):
    targets = list(edges_flips[eee].keys())
    for target in targets:
        adj_flips = get_adj_flips(edges_flips[eee][target])
        if adj_flips is None:
            print(eee, target)
            continue
        for edge in adj_flips:
            u, v = str(edge[0]), str(edge[1])
            if g.has_edge(u, v):
                print(True)

FastGCN_TBS 4080
FastGCN_TBS 19687


In [143]:
output = 'result/test_tran2vec/TBS+WBS_2022_01_11_20_21_43_test.csv'
embeddings = pd.read_csv(output).values # 8w6+
sample_labels = load_labels('dataset/phishing/label.txt') # 890
nodes = list([int(node) for node in sample_labels.keys()])
nodes_labels = list(sample_labels.values())
nodes_embeddings = pd.DataFrame(embeddings[nodes])

In [145]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(nodes_embeddings, nodes_labels, train_size=0.8,
                                                            random_state=2022, stratify=nodes_labels)

In [ ]:
run gf w2v
run gf normal mode
walk cost: 0.09414107799530029 min
w2v cost: 1.7804624001185099 min
classification_report:
              precision    recall  f1-score   support

           0       0.50      0.79      0.61       281
           1       0.87      0.63      0.73       609

    accuracy                           0.68       890
   macro avg       0.68      0.71      0.67       890
weighted avg       0.75      0.68      0.69       890

run_trans2vec, cost:2.068171735604604 min

In [ ]:
run gf w2v
run gf alias mode
walk cost: 0.3224856694539388 min
w2v cost: 1.87216823498408 min
classification_report:
              precision    recall  f1-score   support

           0       0.50      0.82      0.62       272
           1       0.89      0.64      0.74       618

    accuracy                           0.69       890
   macro avg       0.69      0.73      0.68       890
weighted avg       0.77      0.69      0.71       890

run_trans2vec, cost:2.3828219294548036 min

In [ ]:
walk cost: 2.3467689673105876 min
w2v cost: 1.8456719517707825 min
classification_report:
              precision    recall  f1-score   support

           0       0.50      0.79      0.61       281
           1       0.87      0.63      0.73       609

    accuracy                           0.68       890
   macro avg       0.68      0.71      0.67       890
weighted avg       0.75      0.68      0.69       890

run_trans2vec, cost:4.382909286022186 min